# `long` / `wide` — Reference

`long` melts all numeric columns to rows. `wide` pivots a column's values into headers.

| Method | Key params | Purpose |
|---|---|---|
| `.pt.long(c='variable', v='value')` | `c`, `v` | wide → long (melt) |
| `.pt.wide(..., a=None, dropna=True)` | `c`, `v`, `a`, `dropna` | long → wide (pivot); `dropna` only with `a` / pivot_table |

They are designed to work together as a pair, but each is also useful standalone.

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
healthexp = pt.sample_data['healthexp']
flights = pt.sample_data['flights']

In [2]:
penguins

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


## `long` — melt all numeric columns to rows

In [3]:
# Melt all numeric columns to rows — default col/value names
(
    penguins
    .pt.long()
    .sample(10)
)


,species,island,sex,variable,value
1107,Adelie,Torgersen,Male,body_mass_g,4250.0
1044,Adelie,Torgersen,Female,body_mass_g,3200.0
779,Adelie,Dream,Male,flipper_length_mm,205.0
146,Adelie,Dream,Male,bill_length_mm,39.2
669,Gentoo,Biscoe,Male,bill_depth_mm,16.1
859,Chinstrap,Dream,Male,flipper_length_mm,195.0
394,Adelie,Biscoe,Female,bill_depth_mm,17.7
117,Adelie,Torgersen,Male,bill_length_mm,37.3
796,Adelie,Biscoe,Female,flipper_length_mm,181.0
389,Adelie,Dream,Male,bill_depth_mm,18.8


In [4]:
# Custom column names for the variable and value columns
(
    penguins
    .pt.long(c='metric', v='reading')
    .sample(10)
)


,species,island,sex,metric,reading
803,Adelie,Biscoe,Male,flipper_length_mm,196.0
663,Gentoo,Biscoe,Male,bill_depth_mm,16.5
112,Adelie,Biscoe,Female,bill_length_mm,39.7
755,Adelie,Biscoe,Male,flipper_length_mm,188.0
764,Adelie,Torgersen,Female,flipper_length_mm,191.0
325,Gentoo,Biscoe,Male,bill_length_mm,46.8
402,Adelie,Biscoe,Female,bill_depth_mm,16.6
150,Adelie,Dream,Female,bill_length_mm,36.0
1087,Adelie,Biscoe,Male,body_mass_g,3700.0
104,Adelie,Biscoe,Female,bill_length_mm,37.9


In [5]:
# Only non numeric columns survive as id columns — numerics all get melted
# select narrows which numerics get melted
(penguins
 .pt.select('species', 'island', 'bill_length_mm', 'bill_depth_mm')
 .pt.long(c='metric', v='reading')
 .sample(12)
)


,species,island,metric,reading
39,Adelie,Dream,bill_length_mm,39.8
120,Adelie,Torgersen,bill_length_mm,36.2
410,Adelie,Biscoe,bill_depth_mm,16.2
670,Gentoo,Biscoe,bill_depth_mm,14.7
507,Chinstrap,Dream,bill_depth_mm,20.3
629,Gentoo,Biscoe,bill_depth_mm,16.8
185,Chinstrap,Dream,bill_length_mm,51.0
311,Gentoo,Biscoe,bill_length_mm,52.2
65,Adelie,Biscoe,bill_length_mm,41.6
541,Chinstrap,Dream,bill_depth_mm,18.5


## `wide` — pivot a column's values into headers

In [6]:
# flights has one row per year+month — no aggfunc needed
(
    flights
    .pt.wide(c='month', v='passengers')
    .head()
)


,year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,1949,112,118,132,129,121,135,148,148,136,119,104,118
1,1950,115,126,141,135,125,149,170,170,158,133,114,140
2,1951,145,150,178,163,172,178,199,199,184,162,146,166
3,1952,171,180,193,181,183,218,230,242,209,191,172,194
4,1953,196,196,236,235,229,243,264,272,237,211,180,201


In [7]:
# Select only the columns needed so island becomes the clean index
(penguins
 .pt.select('island', 'species', 'body_mass_g')
 .pt.wide(c='species', v='body_mass_g', a='mean')  # dropna=True by default
)


,island,Adelie,Chinstrap,Gentoo
0,Biscoe,3709.659091,NaN,5076.01626
1,Dream,3688.392857,3733.088235,NaN
2,Torgersen,3706.372549,NaN,NaN


`dropna` only applies with `aggfunc` / `pivot_table`. Country `cn` has only a missing balance, so after the pivot that column is all-NA.

In [8]:
import pandas as pd
tiny = pd.DataFrame({
    'id': ['a', 'b', 'b'],
    'country': ['sg', 'sg', 'cn'],
    'balance': [10.0, 20.0, None],
})
tiny

,id,country,balance
0,a,sg,10.0
1,b,sg,20.0
2,b,cn,NaN


In [9]:
tiny.pt.wide(c='country', v='balance', a='mean')  # dropna=True: all-NA column cn dropped

,id,sg
0,a,10.0
1,b,20.0


In [10]:
tiny.pt.wide(c='country', v='balance', a='mean', dropna=False)  # keep cn

,id,cn,sg
0,a,NaN,10.0
1,b,NaN,20.0


## Round-trip: `long` → transform → `wide`
A common pattern: go long to apply uniform operations across all metrics, then come back wide.

In [11]:
# Select only needed columns → long → wide: yields mean bill metrics per island
(penguins
 .pt.select('island', 'bill_length_mm', 'bill_depth_mm')
 .pt.long(c='metric', v='reading')
 .pt.wide(c='metric', v='reading', a='mean')
)


,island,bill_depth_mm,bill_length_mm
0,Biscoe,15.874850,45.257485
1,Dream,18.344355,44.167742
2,Torgersen,18.429412,38.950980
